# Revolving-Credit Default: Model Experiments

This notebook starts from immutable Feature Store Datasets created in notebook 02. It does not inspect raw source facts or rebuild features.

Run in a private Git-backed Snowflake Workspace with Container Runtime CPU 2.9 after bootstrap, verification, source EDA, and feature engineering. Do not run in Snowhouse. No packages need installing.

Every successful fit is logged inside an active Experiment run. Candidate selection is manual, and the controlled-drift hold-out remains inaccessible until the selection checkpoint is deliberately enabled.

## 1. Open the immutable Dataset versions

Paste the Dataset version printed by notebook 02. Selecting one explicit immutable version prevents a rerun from silently training on different data.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4
import json
import sys
import tempfile

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as sf
from snowflake.ml import dataset
from snowflake.ml.feature_store import FeatureStore, CreationMode
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.model import task

session = get_active_session()
config_path = Path("../project.yaml")
if not config_path.is_file():
    config_path = Path("project.yaml")
config = yaml.safe_load(config_path.read_text())
database_name = config["snowflake"]["database"]
dev_schema = config["snowflake"]["schemas"]["dev"]
raw_schema = config["snowflake"]["schemas"]["raw"]
feature_schema = config["snowflake"]["schemas"]["feature_store"]
session.use_role(config["snowflake"]["roles"]["developer"])
session.use_warehouse(config["snowflake"]["warehouse"])
session.use_database(database_name)
session.use_schema(dev_schema)

runtime_versions = {
    name: version(name) for name in ["snowflake-ml-python", "scikit-learn", "numpy", "pandas", "scipy"]
}
display(runtime_versions)

In [ ]:
dataset_version = "PASTE_NOTEBOOK_02_DATASET_VERSION"
assert not dataset_version.startswith("PASTE_"), "Use the immutable version printed by notebook 02."

development_dataset = dataset.load_dataset(
    session, f"{database_name}.{dev_schema}.CREDIT_DEVELOPMENT", version=dataset_version
)
validation_dataset = dataset.load_dataset(
    session, f"{database_name}.{dev_schema}.CREDIT_VALIDATION", version=dataset_version
)
development_df = development_dataset.read.to_snowpark_dataframe(only_feature_cols=False)
validation_df = validation_dataset.read.to_snowpark_dataframe(only_feature_cols=False)
print("Development / validation rows:", development_df.count(), validation_df.count())
print(development_df.columns)

The Dataset schema determines the model inputs. Identifiers, timestamps, labels, and source-control fields are excluded. Scaling, imputation, and categorical encoding remain inside each fitted pipeline because they are model-dependent transformations.

In [ ]:
target_column = "DEFAULT_WITHIN_90D"
numeric_features = [
    "ACCOUNT_AGE_MONTHS", "CUSTOMER_TENURE_MONTHS", "CURRENT_CREDIT_LIMIT",
    "CURRENT_INCOME_ESTIMATE", "LIMIT_TO_INCOME_RATIO", "MONTHS_SINCE_LIMIT_CHANGE",
    "LIMIT_CHANGE_PERCENT_12M", "CURRENT_BALANCE", "CURRENT_UTILISATION",
    "AVG_UTILISATION_30D", "AVG_UTILISATION_90D", "MAX_UTILISATION_90D",
    "UTILISATION_STDDEV_90D", "BALANCE_CHANGE_30D", "BALANCE_CHANGE_90D",
    "DAYS_OVER_LIMIT_30D", "AVAILABLE_CREDIT_RATIO", "PAYMENT_SUM_30D",
    "PAYMENT_SUM_90D", "PAYMENT_COUNT_90D", "PAYMENT_TO_BALANCE_RATIO_30D",
    "PAYMENT_TO_AMOUNT_DUE_RATIO_30D", "PAYMENT_AMOUNT_TREND_3M",
    "MISSED_PAYMENT_COUNT_3M",
    "MISSED_PAYMENT_COUNT_6M", "DAYS_SINCE_LAST_PAYMENT", "CONTACT_COUNT_30D",
    "CONTACT_COUNT_90D", "BROKEN_PROMISE_COUNT_90D", "CURRENT_DAYS_PAST_DUE",
    "MAX_DAYS_PAST_DUE_3M", "MAX_DAYS_PAST_DUE_6M", "DELINQUENT_DAYS_90D",
    "DELINQUENCY_EPISODES_6M", "CONSECUTIVE_DELINQUENT_MONTHS",
    "DAYS_SINCE_LAST_DELINQUENCY", "AVG_UTILISATION_365D",
    "MAX_UTILISATION_365D", "MAX_DAYS_PAST_DUE_365D",
]
categorical_features = ["PRODUCT_CODE", "ORIGINATION_CHANNEL"]
feature_columns = numeric_features + categorical_features
modelling_columns = [
    "ACCOUNT_ID", "OBSERVATION_TS", "OUTCOME_FINALITY_DATE", target_column, *feature_columns,
]
missing_features = sorted(set(feature_columns) - set(development_df.columns))
if missing_features:
    raise ValueError(f"Dataset is missing expected features: {missing_features}")

development_pd = development_df.select(modelling_columns).sort("OBSERVATION_TS", "ACCOUNT_ID").to_pandas()
validation_pd = validation_df.select(modelling_columns).sort("OBSERVATION_TS", "ACCOUNT_ID").to_pandas()
development_pd[numeric_features] = development_pd[numeric_features].astype(float)
validation_pd[numeric_features] = validation_pd[numeric_features].astype(float)
development_pd[target_column] = development_pd[target_column].astype(int)
validation_pd[target_column] = validation_pd[target_column].astype(int)
training_features = development_pd[feature_columns]
training_labels = development_pd[target_column]
validation_features = validation_pd[feature_columns]
validation_labels = validation_pd[target_column]
training_sample = development_df.select(feature_columns).limit(50)
calibration_bins = 10
display(development_pd[modelling_columns].isna().sum().to_frame("DEVELOPMENT_NULLS"))

## 5. Establish a Reference Before Fitting

After checking missingness, compare against a constant probability equal to development prevalence. This reference does not fit an estimator. Its ROC AUC should be 0.5, and its average precision should equal validation prevalence. A useful model should improve ranking without sacrificing probability quality.

We will use ROC AUC, average precision, and Brier score throughout. Accuracy is not a primary metric for this imbalanced target.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.calibration import CalibrationDisplay

reference_probability = np.full(len(validation_labels), training_labels.mean())
reference_metrics = {
    "ROC_AUC": roc_auc_score(validation_labels, reference_probability),
    "AVERAGE_PRECISION": average_precision_score(validation_labels, reference_probability),
    "BRIER_SCORE": brier_score_loss(validation_labels, reference_probability),
}
pd.DataFrame([reference_metrics], index=["Development prevalence reference"])

## 6. Start With a Linear Model

The prevalence reference shows the difficulty of the ranking problem. Begin with logistic regression because it gives a simple, inspectable hypothesis: each feature changes log-odds additively.

This decision introduces preprocessing for the first time. Numeric fields use different units, so standardise them for the regularised linear model. Categorical fields need one-hot encoding; unknown categories are ignored so a later month does not fail solely because it contains a new level. We leave class weighting off because changing class weights also changes the probability interpretation we want to inspect.

At this point we also create names for the Experiment and model collection. They describe artefacts discovered during this notebook, not platform objects that had to be known during bootstrap. Each fit and its model log share an active Experiment run. The Snowpark `training_sample` is used for signature inference so the registered pipeline retains lineage to the Dataset.

[Experiments](https://docs.snowflake.com/en/developer-guide/snowflake-ml/experiments), [Registry pipeline support](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/built-in-models/scikit-learn), [ML lineage](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-lineage).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

seed = 20260912
experiment_name = "CREDIT_DEFAULT_DEVELOPMENT"
model_name = "CREDIT_DEFAULT_EARLY_WARNING"
version_prefix = "CANDIDATE"

exp = ExperimentTracking(
    session=session,
    database_name=database_name,
    schema_name=dev_schema,
)
exp.set_experiment(experiment_name)

model_dependencies = [
    f"{name}=={runtime_versions[name]}"
    for name in ["scikit-learn", "numpy", "pandas", "scipy"]
]
python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
experiment_inputs = {
    "development_dataset": development_dataset.fully_qualified_name,
    "validation_dataset": validation_dataset.fully_qualified_name,
    "dataset_version": dataset_version,
    "features": json.dumps(feature_columns),
    "seed": seed,
    "development_cutoff": str(development_pd["OBSERVATION_TS"].max()),
    "validation_cutoff": str(validation_pd["OBSERVATION_TS"].max()),
    "runtime_packages": json.dumps(runtime_versions),
    "python": python_version,
}

linear_model = Pipeline([
    ("features", ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_features),
        ("categorical", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ), categorical_features),
    ])),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=seed,
    )),
])
linear_model

In [ ]:
linear_run = "LINEAR_" + uuid4().hex[:12].upper()
with exp.start_run(linear_run):
    exp.log_params({**experiment_inputs, "family": "LOGISTIC", "C": 1.0, "max_iter": 1000})
    linear_model.fit(training_features, training_labels)
    linear_probability = linear_model.predict_proba(validation_features)[:, list(linear_model.classes_).index(1)]
    linear_metrics = {
        "validation_roc_auc": roc_auc_score(validation_labels, linear_probability),
        "validation_average_precision": average_precision_score(validation_labels, linear_probability),
        "validation_brier_score": brier_score_loss(validation_labels, linear_probability),
    }
    exp.log_metrics(linear_metrics)
    linear_registered = exp.log_model(
        linear_model, model_name=model_name, version_name = f"{version_prefix}_{linear_run}",
        sample_input_data=training_sample, task=task.Task.TABULAR_BINARY_CLASSIFICATION,
        target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
        conda_dependencies=model_dependencies, python_version=python_version,
        options={"relax_version": False, "target_methods": ["predict", "predict_proba"]},
        comment="Portfolio monitoring: initial logistic regression experiment.",
    )
pd.Series(linear_metrics, name="Logistic validation").to_frame()

The metrics tell us whether ranking improves on prevalence, but not how the model does it. Inspect coefficients and calibration next. Numeric coefficients are per scaled unit; categorical coefficients are relative to the fitted encoding, not causal effects. Does the model rely on recent payment behaviour as expected? Is origination channel adding signal or just noise?

In [ ]:
coefficients = pd.Series(linear_model.named_steps["classifier"].coef_[0],
                         index=linear_model.named_steps["features"].get_feature_names_out())
coefficients.sort_values().plot.barh(figsize=(9, 7), title="Logistic coefficients after numeric scaling")
plt.tight_layout()
plt.show()
CalibrationDisplay.from_predictions(validation_labels, linear_probability, n_bins=10, strategy="uniform")
plt.title("Logistic validation calibration")
plt.show()

## 7. Test a Non-Linear Alternative

The linear model establishes a reference for both ranking and calibration. Histogram gradient boosting now tests whether interactions and non-linear responses improve it. Use exactly the same Dataset versions. Numeric scaling is unnecessary; categorical encoding remains part of the pipeline.

Disable automatic early stopping because its random internal split would not match our temporal validation protocol. We will investigate iteration count during HPO rather than use held-out data.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

tree_model = Pipeline([
    ("features", ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ])),
    ("classifier", HistGradientBoostingClassifier(learning_rate=0.08, max_iter=120,
        max_leaf_nodes=15, l2_regularization=1.0, early_stopping=False, random_state=seed)),
])
tree_model

In [ ]:
tree_run = "HIST_" + uuid4().hex[:12].upper()
with exp.start_run(tree_run):
    exp.log_params({**experiment_inputs, "family": "HIST", "learning_rate": 0.08,
                   "max_iter": 120, "max_leaf_nodes": 15, "l2_regularization": 1.0, "early_stopping": False})
    tree_model.fit(training_features, training_labels)
    tree_probability = tree_model.predict_proba(validation_features)[:, list(tree_model.classes_).index(1)]
    tree_metrics = {
        "validation_roc_auc": roc_auc_score(validation_labels, tree_probability),
        "validation_average_precision": average_precision_score(validation_labels, tree_probability),
        "validation_brier_score": brier_score_loss(validation_labels, tree_probability),
    }
    exp.log_metrics(tree_metrics)
    tree_registered = exp.log_model(
        tree_model, model_name=model_name, version_name=f"{version_prefix}_{tree_run}",
        sample_input_data=training_sample, task=task.Task.TABULAR_BINARY_CLASSIFICATION,
        target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
        conda_dependencies=model_dependencies, python_version=python_version,
        options={"relax_version": False, "target_methods": ["predict", "predict_proba"]},
        comment="Portfolio monitoring: initial histogram boosting experiment.",
    )
comparison = pd.DataFrame([linear_metrics, tree_metrics], index=["Logistic", "Histogram boosting"])
comparison

## 8. Compare How the Models Behave

Do the two candidates rank the same risky accounts, and do their probabilities mean the same thing? Read precision-recall and calibration together. The score distributions show whether apparent separation depends on a small tail. These cells use existing predictions; they do not retrain models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, probabilities in [("Logistic", linear_probability), ("Histogram boosting", tree_probability)]:
    PrecisionRecallDisplay.from_predictions(validation_labels, probabilities, name=name, ax=axes[0])
    RocCurveDisplay.from_predictions(validation_labels, probabilities, name=name, ax=axes[1])
    CalibrationDisplay.from_predictions(validation_labels, probabilities, n_bins=10, strategy="uniform", name=name, ax=axes[2])
plt.tight_layout()
plt.show()

validation_scores = validation_pd[["ACCOUNT_ID", "OBSERVATION_TS", target_column] + categorical_features].copy()
validation_scores["LOGISTIC"] = linear_probability
validation_scores["HIST"] = tree_probability
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for model_column, axis in zip(["LOGISTIC", "HIST"], axes):
    sns.histplot(data=validation_scores, x=model_column, hue=target_column, bins=30,
                 stat="density", common_norm=False, element="step", ax=axis)
plt.tight_layout()
plt.show()

### Translate Ranking Into a Review Queue

The plots show ranking and calibration, but the intended use is case prioritisation. Assume for this experiment that operations can review 10% of active accounts each month. This is a notebook scenario choice, not a preconfigured platform rule or a real credit policy.

Rank within each observation month, allocate `ceil(account_count * review_rate)` reviews, and break score ties by account ID so row ordering does not change the result. Recall is undefined when a month has no defaults.

The monthly table exposes the numerator, denominator, and workload. Are gains present across months rather than concentrated in one?

In [ ]:
review_rate = 0.10
monthly_results = []
for model_column in ["LOGISTIC", "HIST"]:
    for month, cohort in validation_scores.groupby("OBSERVATION_TS"):
        review_count = int(np.ceil(len(cohort) * review_rate))
        # Stable tie-breaking makes the review queue reproducible when scores tie.
        reviews = cohort.sort_values(
            [model_column, "ACCOUNT_ID"],
            ascending=[False, True],
        ).head(review_count)
        default_count = int(cohort[target_column].sum())
        captured = int(reviews[target_column].sum())
        monthly_results.append({
            "MODEL": model_column,
            "MONTH": month,
            "ACCOUNTS": len(cohort),
            "REVIEWS": review_count,
            "DEFAULTS": default_count,
            "CAPTURED": captured,
            "RECALL": captured / default_count if default_count else np.nan,
        })

monthly_review = pd.DataFrame(monthly_results)
display(monthly_review)
sns.lineplot(
    data=monthly_review,
    x="MONTH",
    y="RECALL",
    hue="MODEL",
    marker="o",
)
plt.title("Validation defaults captured within monthly review capacity")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Check Segments and Feature Reliance

Next, inspect support and ranking within the three operational segments. A missing or single-class segment does not provide evidence of acceptable performance. These are account-month counts, not independent borrowers.

Permutation importance on a bounded validation sample tests which inputs the tree actually uses. Correlated features can substitute for one another; this is diagnostic evidence, not a causal interpretation. Finish feature investigations before HPO and do not consult the 2026 hold-out.

In [ ]:
segment_results = []
for segment in categorical_features:
    for value, cohort in validation_scores.groupby(segment):
        positives = int(cohort[target_column].sum())
        negatives = len(cohort) - positives
        for model_column in ["LOGISTIC", "HIST"]:
            segment_results.append({"SEGMENT": segment, "VALUE": value, "MODEL": model_column,
                "OBSERVATIONS": len(cohort), "POSITIVES": positives, "NEGATIVES": negatives,
                "ROC_AUC": roc_auc_score(cohort[target_column], cohort[model_column]) if positives and negatives else np.nan})
segment_comparison = pd.DataFrame(segment_results)
display(segment_comparison)

from sklearn.inspection import permutation_importance
importance_sample = validation_pd.sample(n=min(3000, len(validation_pd)), random_state=seed)
importance = permutation_importance(tree_model, importance_sample[feature_columns], importance_sample[target_column],
    scoring="average_precision", n_repeats=3, random_state=seed, n_jobs=1)
pd.Series(importance.importances_mean, index=feature_columns).sort_values().plot.barh(
    title="Tree model: average-precision decrease after permutation", figsize=(8, 5))
plt.tight_layout()
plt.show()

## 9. Tune Only After Understanding the Initial Models

**Pause here and write investigation notes in this Markdown cell before running HPO.** Record what the plots suggest about feature reliance, probability calibration, monthly stability, and segments. If those observations change the feature hypothesis, make another explicit tracked model attempt before tuning.

The initial candidates now give us a reason to ask a narrower question: can modest regularisation or tree-complexity changes improve validation average precision without damaging Brier score? Choose a small budget of four trials per family because this is an interactive experiment, not an exhaustive search.

The Snowflake Tuner needs a training function because trials run outside the notebook's local process. That function contains only the work that must happen remotely. Each one-trial Tuner call remains inside an Experiment run started by the notebook, and the returned fitted pipeline is logged before the run ends. `report(model=...)` checkpoints a Tuner result; it does not replace Experiment model logging.

Running trials sequentially is deliberate: we are not assuming the notebook's active Experiment context propagates to parallel workers. [Snowflake HPO](https://docs.snowflake.com/en/developer-guide/snowflake-ml/container-hpo).

In [ ]:
from sklearn.base import clone
from snowflake.ml.data.data_connector import DataConnector
from snowflake.ml.modeling.tune import (
    Tuner,
    TunerConfig,
    get_tuner_context,
    loguniform,
    randint,
)
from snowflake.ml.modeling.tune.search import RandomSearch

trials_per_family = 4
trial_budgets = {
    "LOGISTIC": trials_per_family,
    "HIST": trials_per_family,
}
search_spaces = {
    "LOGISTIC": {
        "C": loguniform(0.01, 10.0),
    },
    "HIST": {
        "learning_rate": loguniform(0.03, 0.15),
        "max_leaf_nodes": randint(7, 32),
        "max_iter": randint(60, 201),
        "l2_regularization": loguniform(0.1, 10.0),
    },
}
base_models = {
    "LOGISTIC": linear_model,
    "HIST": tree_model,
}
# DataConnectors let Tuner workers read the two immutable Dataset-backed frames.
dataset_map = {
    "development": DataConnector.from_dataframe(
        development_df.select(modelling_columns)
    ),
    "validation": DataConnector.from_dataframe(
        validation_df.select(modelling_columns)
    ),
}

pd.DataFrame({
    "FAMILY": list(trial_budgets),
    "TRIALS": list(trial_budgets.values()),
    "QUESTION": [
        "How much linear regularisation?",
        "How much non-linear complexity?",
    ],
})

In [ ]:
def train_trial():
    context = get_tuner_context()
    parameters = context.get_hyper_params()
    trial_data = context.get_dataset_map()

    # Tuner workers receive DataConnectors, not notebook-local pandas objects.
    training = trial_data["development"].to_pandas().sort_values(
        ["OBSERVATION_TS", "ACCOUNT_ID"]
    )
    validation = trial_data["validation"].to_pandas().sort_values(
        ["OBSERVATION_TS", "ACCOUNT_ID"]
    )
    training[numeric_features] = training[numeric_features].astype(float)
    validation[numeric_features] = validation[numeric_features].astype(float)

    # Clone keeps every trial independent while preserving the family pipeline.
    trial_model = clone(trial_template)
    trial_model.set_params(**{
        "classifier__" + name: value
        for name, value in parameters.items()
    })
    trial_model.fit(
        training[feature_columns],
        training[target_column].astype(int),
    )
    probability = trial_model.predict_proba(
        validation[feature_columns]
    )[:, list(trial_model.classes_).index(1)]

    metrics = {
        "validation_average_precision": average_precision_score(
            validation[target_column], probability
        ),
        "validation_roc_auc": roc_auc_score(
            validation[target_column], probability
        ),
        "validation_brier_score": brier_score_loss(
            validation[target_column], probability
        ),
    }
    # Returning the model avoids a second, hidden fit when the driver logs it.
    context.report(metrics=metrics, model=trial_model)

In [ ]:
hpo_results = []
hpo_models = {}
family = "LOGISTIC"
trial_template = base_models[family]
for attempt in range(trial_budgets[family]):
    run_name = "HPO_LINEAR_" + uuid4().hex[:12].upper()
    with exp.start_run(run_name):
        exp.log_params({**experiment_inputs, "family": family, "search_seed": seed + attempt})
        tuner = Tuner(train_trial, search_spaces[family], TunerConfig(
            metric="validation_average_precision", mode="max", search_alg=RandomSearch(random_state=seed + attempt),
            num_trials=1, max_concurrent_trials=1,
        ))
        result = tuner.run(dataset_map=dataset_map)
        trained_model = result.best_model
        if trained_model is None:
            raise RuntimeError("The trial did not return a fitted model; stop before further tuning.")
        actual_parameters = {name: trained_model.named_steps["classifier"].get_params()[name] for name in search_spaces[family]}
        metrics = {name: float(result.best_result.iloc[0][name]) for name in linear_metrics}
        exp.log_params(actual_parameters)
        exp.log_metrics(metrics)
        registered = exp.log_model(
            trained_model, model_name=model_name, version_name=f"{version_prefix}_{run_name}",
            sample_input_data=training_sample, task=task.Task.TABULAR_BINARY_CLASSIFICATION,
            target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
            conda_dependencies=model_dependencies, python_version=python_version,
            options={"relax_version": False, "target_methods": ["predict", "predict_proba"]},
        )
    hpo_results.append({"RUN": run_name, "FAMILY": family, "VERSION": registered.version_name,
                        "PARAMETERS": json.dumps(actual_parameters), **metrics})
    hpo_models[run_name] = trained_model
    display(pd.DataFrame(hpo_results).tail(1))

Inspect the logistic trials and their linked Registry versions before continuing. Is regularisation materially changing validation ranking or Brier score? The next cell applies the same small-budget experiment to the tree model. It does not reuse the linear search space or increase the budget automatically.

In [ ]:
family = "HIST"
trial_template = base_models[family]
for attempt in range(trial_budgets[family]):
    run_name = "HPO_HIST_" + uuid4().hex[:12].upper()
    with exp.start_run(run_name):
        exp.log_params({**experiment_inputs, "family": family, "search_seed": seed + 100 + attempt})
        tuner = Tuner(train_trial, search_spaces[family], TunerConfig(
            metric="validation_average_precision", mode="max", search_alg=RandomSearch(random_state=seed + 100 + attempt),
            num_trials=1, max_concurrent_trials=1,
        ))
        result = tuner.run(dataset_map=dataset_map)
        trained_model = result.best_model
        if trained_model is None:
            raise RuntimeError("The trial did not return a fitted model; stop before further tuning.")
        actual_parameters = {name: trained_model.named_steps["classifier"].get_params()[name] for name in search_spaces[family]}
        metrics = {name: float(result.best_result.iloc[0][name]) for name in tree_metrics}
        exp.log_params(actual_parameters)
        exp.log_metrics(metrics)
        registered = exp.log_model(
            trained_model, model_name=model_name, version_name=f"{version_prefix}_{run_name}",
            sample_input_data=training_sample, task=task.Task.TABULAR_BINARY_CLASSIFICATION,
            target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
            conda_dependencies=model_dependencies, python_version=python_version,
            options={"relax_version": False, "target_methods": ["predict", "predict_proba"]},
        )
    hpo_results.append({"RUN": run_name, "FAMILY": family, "VERSION": registered.version_name,
                        "PARAMETERS": json.dumps(actual_parameters), **metrics})
    hpo_models[run_name] = trained_model
    display(pd.DataFrame(hpo_results).tail(1))

In [ ]:
candidate_table = pd.DataFrame([
    {"RUN": linear_run, "FAMILY": "LOGISTIC", "VERSION": linear_registered.version_name, "PARAMETERS": "Initial linear model", **linear_metrics},
    {"RUN": tree_run, "FAMILY": "HIST", "VERSION": tree_registered.version_name, "PARAMETERS": "Initial tree model", **tree_metrics},
    *hpo_results,
])
display(candidate_table.sort_values("validation_average_precision", ascending=False))
sns.scatterplot(data=candidate_table, x="validation_brier_score", y="validation_average_precision", hue="FAMILY")
plt.title("Validation trade-off: lower Brier, higher average precision")
plt.show()
models_by_run = {linear_run: linear_model, tree_run: tree_model, **hpo_models}

## 10. Choose a Candidate From the Evidence

The table includes the initial models as well as every successfully logged HPO trial. The highest average precision is a starting point, not an automatic promotion rule. Consider ROC AUC, Brier score, complexity, and the earlier monthly/segment evidence. Repeated tuning makes validation estimates optimistic; the untouched hold-out is still needed.

Set `selected_run` to one displayed run and write a short rationale. There is intentionally no default winner. Inspect the selected candidate's validation calibration in the next cell before deciding to reveal held-out outcomes.

In [ ]:
selected_run = None
selection_reason = ""

assert selected_run in models_by_run, "Choose a run from candidate_table before continuing."
assert selection_reason.strip(), "Record the validation evidence supporting this choice."
selected_model = models_by_run[selected_run]
selected_candidate = candidate_table.loc[candidate_table["RUN"] == selected_run].iloc[0]
selected_probability = selected_model.predict_proba(validation_features)[:, list(selected_model.classes_).index(1)]
display(selected_candidate.to_frame("Selected candidate"))
CalibrationDisplay.from_predictions(validation_labels, selected_probability,
    n_bins=calibration_bins, strategy="uniform")
plt.title("Selected candidate: validation calibration")
plt.show()

## 11. Freeze the Decision Before Revealing the Hold-Out

The next stage is one final evaluation, not another tuning cycle. Only now define the demonstration gates, segment support expectations, calibration resolution, and bootstrap size. These choices must be recorded before held-out outcomes are scored, but they did not need to exist when the platform environment was provisioned.

Refit uses only the eligible development and validation rows already inspected; it does not add the purged boundary months. The current synthetic panel contains the same accounts each month, so this is forward-time existing-account evaluation, not an independent new-borrower test.

Change `evaluate_holdout` deliberately after reviewing the selected run and the choices below. The kernel flag is a reminder, not durable access control: restarting the kernel does not make previously seen outcomes unseen. If the gates fail, preserve that result rather than change thresholds or try another model against the same hold-out.

In [ ]:
evaluation_gates = {
    "minimum_roc_auc": 0.72,
    "minimum_average_precision": 0.20,
    "maximum_brier_score": 0.16,
    "minimum_segment_roc_auc": 0.62,
}
segment_support = {
    "minimum_observations": 200,
    "minimum_positives": 20,
    "minimum_negatives": 20,
}
calibration_bins = 10
bootstrap_replicates = 200

evaluate_holdout = False
assert evaluate_holdout, "Review the selected candidate and frozen choices first."
assert not globals().get("holdout_revealed", False), (
    "This kernel has already revealed the hold-out; "
    "do not use it for further model selection."
)

fs = FeatureStore(
    session=session,
    database=database_name,
    name=feature_schema,
    default_warehouse=config["snowflake"]["warehouse"],
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)
feature_version = "V01"
feature_views = [
    fs.get_feature_view(name, feature_version)
    for name in (
        "ACCOUNT_PROFILE_FV", "ACCOUNT_BALANCE_FV", "ACCOUNT_PAYMENT_FV",
        "ACCOUNT_CONTACT_FV", "ACCOUNT_DELINQUENCY_FV", "ACCOUNT_AGGREGATION_FV",
    )
]
labels = session.table(f"{database_name}.{raw_schema}.AVAILABLE_GROUND_TRUTH").select(
    "ACCOUNT_ID",
    sf.col("OBSERVATION_DATE").cast("timestamp_ntz").alias("OBSERVATION_TS"),
    "OUTCOME_FINALITY_DATE",
    target_column,
)
refit_spine = labels.filter(
    (sf.col("OBSERVATION_TS") < sf.lit(config["data"]["drift_start_date"]))
    & (sf.col("OUTCOME_FINALITY_DATE") < sf.lit(config["data"]["drift_start_date"]))
)
refit_dataset = fs.generate_dataset(
    name=f"{database_name}.{dev_schema}.CREDIT_REFIT",
    version=dataset_version,
    spine_df=refit_spine,
    features=feature_views,
    spine_timestamp_col="OBSERVATION_TS",
    spine_label_cols=[target_column],
    include_feature_view_timestamp_col=True,
    join_method="cte",
    desc="Development and validation observations for the frozen-candidate refit.",
)
refit_df = refit_dataset.read.to_snowpark_dataframe(only_feature_cols=False)
refit_pd = refit_df.select(modelling_columns).sort(
    "OBSERVATION_TS", "ACCOUNT_ID"
).to_pandas()
refit_pd[numeric_features] = refit_pd[numeric_features].astype(float)
refit_pd[target_column] = refit_pd[target_column].astype(int)
refit_sample = refit_df.select(feature_columns).limit(50)

display(pd.Series(evaluation_gates, name="Frozen evaluation gates").to_frame())
display(pd.Series(segment_support, name="Frozen support thresholds").to_frame())
print("Refit observations:", len(refit_pd), "Selected run:", selected_run)

### Refit the Selected Candidate in a Closed Experiment Run

The refit is a distinct tracked fit. A context-managed run ensures that fitting or Registry failure cannot leave stale Experiment state. Held-out evidence is computed in the following cells without changing the fitted model or frozen gates; the conclusion is recorded in a second context-managed run.

In [ ]:
final_run = "FINAL_" + uuid4().hex[:12].upper()
final_model = clone(selected_model)
with exp.start_run(final_run):
    exp.log_params({
        **experiment_inputs,
        "stage": "FINAL_REFIT",
        "selected_run": selected_run,
        "selected_model_version": selected_candidate["VERSION"],
        "selection_reason": selection_reason,
        "refit_dataset": refit_dataset.fully_qualified_name,
        "gates": json.dumps(evaluation_gates),
        "parameters": str(selected_model.named_steps["classifier"].get_params()),
    })
    final_model.fit(refit_pd[feature_columns], refit_pd[target_column])
    refit_registered = exp.log_model(
        final_model,
        model_name=model_name,
        version_name=f"{version_prefix}_{final_run}_REFIT",
        sample_input_data=refit_sample,
        task=task.Task.TABULAR_BINARY_CLASSIFICATION,
        target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
        conda_dependencies=model_dependencies,
        python_version=python_version,
        options={"relax_version": False, "target_methods": ["predict", "predict_proba"]},
        comment="Refitted selected notebook candidate before one held-out evaluation.",
    )
print("Refit complete and run closed:", final_run, refit_registered.version_name)

In [ ]:
held_out_spine = labels.filter(
    sf.col("OBSERVATION_TS") >= sf.lit(config["data"]["drift_start_date"])
)
held_out_dataset = fs.generate_dataset(
    name=f"{database_name}.{dev_schema}.CREDIT_HELD_OUT",
    version=dataset_version,
    spine_df=held_out_spine,
    features=feature_views,
    spine_timestamp_col="OBSERVATION_TS",
    spine_label_cols=[target_column],
    include_feature_view_timestamp_col=True,
    join_method="cte",
    desc="Controlled-drift observations reserved for one evaluation after model selection.",
)
held_out_df = held_out_dataset.read.to_snowpark_dataframe(only_feature_cols=False)
held_out_count = held_out_df.count()
held_out_key_count = held_out_df.select(
    "ACCOUNT_ID", "OBSERVATION_TS"
).distinct().count()
timestamp_columns = [column for column in held_out_df.columns if column.endswith("_TS") and column != "OBSERVATION_TS"]
late_feature_condition = sf.lit(False)
for timestamp_column in timestamp_columns:
    late_feature_condition = late_feature_condition | (sf.col(timestamp_column) > sf.col("OBSERVATION_TS"))
late_feature_count = held_out_df.filter(late_feature_condition).count()
print("Held-out rows / distinct keys / late timestamps:", held_out_count, held_out_key_count, late_feature_count)
if held_out_count != held_out_key_count or late_feature_count:
    raise ValueError("Investigate held-out Dataset integrity before revealing outcomes.")
holdout_revealed = True

held_out_pd = held_out_df.select(modelling_columns).sort(
    "OBSERVATION_TS", "ACCOUNT_ID"
).to_pandas()
held_out_pd[numeric_features] = held_out_pd[numeric_features].astype(float)
held_out_pd[target_column] = held_out_pd[target_column].astype(int)
print("Held-out observations:", len(held_out_pd))
print(
    "Accounts unseen during fitting:",
    len(set(held_out_pd["ACCOUNT_ID"]) - set(refit_pd["ACCOUNT_ID"])),
)
held_out_pd.isna().sum().to_frame("HELD_OUT_NULLS")

Check the retrieved timestamps and missingness before scoring. The same-month mismatch count should still be zero. The following comparison separates validation evidence for the original fitted candidate from held-out evidence for its refitted version. Differences reflect time, controlled drift, and the larger refit population, not just one cause.

In [ ]:
held_out_probability = final_model.predict_proba(held_out_pd[feature_columns])[:, list(final_model.classes_).index(1)]
held_out_labels = held_out_pd[target_column]
held_out_metrics = {
    "held_out_roc_auc": roc_auc_score(held_out_labels, held_out_probability),
    "held_out_average_precision": average_precision_score(held_out_labels, held_out_probability),
    "held_out_brier_score": brier_score_loss(held_out_labels, held_out_probability),
}
display(pd.DataFrame({"Validation candidate": [selected_candidate[name] for name in linear_metrics],
                      "Held-out refit": list(held_out_metrics.values())}, index=["ROC AUC", "Average precision", "Brier score"]))
CalibrationDisplay.from_predictions(held_out_labels, held_out_probability, n_bins=calibration_bins, strategy="uniform")
plt.title("Final held-out calibration under controlled drift")
plt.show()
held_out_scores = held_out_pd.copy()
held_out_scores["PROBABILITY"] = held_out_probability
held_out_scores["BIN"] = pd.cut(held_out_probability, np.linspace(0, 1, calibration_bins + 1), include_lowest=True)
calibration_table = held_out_scores.groupby("BIN", observed=True).agg(
    OBSERVATIONS=(target_column, "size"), MEAN_PROBABILITY=("PROBABILITY", "mean"), OBSERVED_DEFAULT_RATE=(target_column, "mean"))
calibration_table

### Does the Review Workload Still Find Defaults?

Repeat the same monthly review calculation used on validation. Check account counts and captured defaults alongside recall; a changing prevalence can make recall and precision move differently. Do not change the review rate after inspecting these results.

In [ ]:
held_out_months = []
for month, cohort in held_out_scores.groupby("OBSERVATION_TS"):
    review_count = int(np.ceil(len(cohort) * review_rate))
    reviews = cohort.sort_values(["PROBABILITY", "ACCOUNT_ID"], ascending=[False, True]).head(review_count)
    default_count = int(cohort[target_column].sum())
    captured = int(reviews[target_column].sum())
    held_out_months.append({"MONTH": month, "ACCOUNTS": len(cohort), "REVIEWS": review_count,
        "DEFAULTS": default_count, "CAPTURED": captured, "RECALL": captured / default_count if default_count else np.nan})
held_out_monthly = pd.DataFrame(held_out_months)
display(held_out_monthly)
held_out_monthly.plot(x="MONTH", y="RECALL", marker="o", title="Held-out monthly recall at fixed review capacity")
plt.show()
if held_out_monthly["RECALL"].notna().any():
    held_out_monthly_summary = {
        "held_out_mean_monthly_recall": float(held_out_monthly["RECALL"].mean()),
        "held_out_worst_monthly_recall": float(held_out_monthly["RECALL"].min()),
    }
else:
    held_out_monthly_summary = {}

### Check Support Before Interpreting Segment Scores

An overall result can conceal a weak product or tenure segment. Compare each expected validation segment with the held-out population. Missing segments, too few observations, or too few positives/negatives are insufficient evidence, not passing scores. These thresholds are demonstration choices, not real credit-risk policy.

In [ ]:
held_out_segments = []
for segment in categorical_features:
    expected_values = sorted(
        set(validation_pd[segment].dropna().unique())
        | set(held_out_scores[segment].dropna().unique())
    )
    for value in expected_values:
        cohort = held_out_scores.loc[held_out_scores[segment] == value]
        positives = int(cohort[target_column].sum())
        negatives = len(cohort) - positives
        sufficient = (
            len(cohort) >= segment_support["minimum_observations"]
            and positives >= segment_support["minimum_positives"]
            and negatives >= segment_support["minimum_negatives"]
        )
        score = (
            roc_auc_score(cohort[target_column], cohort["PROBABILITY"])
            if positives and negatives
            else np.nan
        )
        held_out_segments.append({
            "SEGMENT": segment,
            "VALUE": value,
            "PRESENT_IN_VALIDATION": value in set(validation_pd[segment].dropna().unique()),
            "PRESENT_IN_HELD_OUT": len(cohort) > 0,
            "OBSERVATIONS": len(cohort),
            "POSITIVES": positives,
            "NEGATIVES": negatives,
            "ROC_AUC": score,
            "SUFFICIENT_SUPPORT": sufficient,
            "MEETS_FLOOR": (
                sufficient
                and score >= evaluation_gates["minimum_segment_roc_auc"]
            ),
        })

held_out_segment_table = pd.DataFrame(held_out_segments)
held_out_segment_table

### How Much Uncertainty Is There?

Repeated account-month rows are correlated. Resample whole accounts, retaining all their months, rather than treating each row as independent. These descriptive intervals reflect account sampling for this fixed time period; they do not measure uncertainty about future economic regimes. They are not another acceptance gate.

In [ ]:
account_rows = held_out_scores.groupby("ACCOUNT_ID").indices
account_ids = list(account_rows)
rng = np.random.default_rng(seed)
bootstrap_metrics = []
for replicate in range(bootstrap_replicates):
    sampled_accounts = rng.choice(account_ids, size=len(account_ids), replace=True)
    row_positions = np.concatenate([account_rows[account] for account in sampled_accounts])
    sample = held_out_scores.iloc[row_positions]
    if sample[target_column].nunique() == 2:
        bootstrap_metrics.append({
            "ROC_AUC": roc_auc_score(sample[target_column], sample["PROBABILITY"]),
            "AVERAGE_PRECISION": average_precision_score(sample[target_column], sample["PROBABILITY"]),
            "BRIER_SCORE": brier_score_loss(sample[target_column], sample["PROBABILITY"]),
        })
uncertainty = pd.DataFrame(bootstrap_metrics).quantile([0.025, 0.5, 0.975]).T
uncertainty.columns = ["LOW_95", "MEDIAN", "HIGH_95"]
uncertainty

## 12. Record the Experimental Conclusion

Compare the frozen gates with the observed results. A rejected model remains useful evidence. Passing these demonstration gates does not approve a real credit model, promote an alias, or deploy inference.

Write the conclusion after reviewing calibration, monthly performance, segments, and uncertainty. The next cell logs the held-out evidence in its own context-managed run linked to the already logged refit model. Do not change the gates to make the candidate pass.

In [ ]:
supported_segments = held_out_segment_table.loc[
    held_out_segment_table["SUFFICIENT_SUPPORT"]
]
unsupported_segment_count = int(
    (~held_out_segment_table["SUFFICIENT_SUPPORT"]).sum()
)
segment_floor_observed = (
    supported_segments["ROC_AUC"].min()
    if len(supported_segments)
    else np.nan
)
segment_floor_pass = (
    len(supported_segments) > 0
    and supported_segments["MEETS_FLOOR"].all()
)

gate_table = pd.DataFrame([
    {
        "CHECK": "ROC AUC",
        "OBSERVED": held_out_metrics["held_out_roc_auc"],
        "THRESHOLD": evaluation_gates["minimum_roc_auc"],
        "PASS": held_out_metrics["held_out_roc_auc"]
        >= evaluation_gates["minimum_roc_auc"],
    },
    {
        "CHECK": "Average precision",
        "OBSERVED": held_out_metrics["held_out_average_precision"],
        "THRESHOLD": evaluation_gates["minimum_average_precision"],
        "PASS": held_out_metrics["held_out_average_precision"]
        >= evaluation_gates["minimum_average_precision"],
    },
    {
        "CHECK": "Brier score (maximum)",
        "OBSERVED": held_out_metrics["held_out_brier_score"],
        "THRESHOLD": evaluation_gates["maximum_brier_score"],
        "PASS": held_out_metrics["held_out_brier_score"]
        <= evaluation_gates["maximum_brier_score"],
    },
    {
        "CHECK": "All required segments have support",
        "OBSERVED": unsupported_segment_count,
        "THRESHOLD": 0,
        "PASS": unsupported_segment_count == 0,
    },
    {
        "CHECK": "Supported segment ROC AUC floor",
        "OBSERVED": segment_floor_observed,
        "THRESHOLD": evaluation_gates["minimum_segment_roc_auc"],
        "PASS": bool(segment_floor_pass),
    },
])
experimental_status = (
    "ACCEPTED" if gate_table["PASS"].all() else "REJECTED"
)
display(gate_table)
print("Segments without enough evidence:", unsupported_segment_count)
print("Experimental gate result:", experimental_status)

In [ ]:
experiment_conclusion = ""
assert experiment_conclusion.strip(), "Record what the held-out evidence means before logging the conclusion."
conclusion_run = "CONCLUSION_" + uuid4().hex[:12].upper()
with exp.start_run(conclusion_run):
    exp.log_params({
        **experiment_inputs,
        "stage": "HELD_OUT_CONCLUSION",
        "refit_run": final_run,
        "refit_model_version": refit_registered.version_name,
        "held_out_dataset": held_out_dataset.fully_qualified_name,
        "experimental_status": experimental_status,
        "conclusion": experiment_conclusion,
    })
    exp.log_metrics({**held_out_metrics, **held_out_monthly_summary})
    with tempfile.TemporaryDirectory() as directory:
        for name, table in [
            ("monthly_review", held_out_monthly),
            ("segments", held_out_segment_table),
            ("calibration", calibration_table.reset_index().astype({"BIN": str})),
            ("uncertainty", uncertainty.reset_index()),
            ("gates", gate_table),
        ]:
            path = Path(directory) / (name + ".csv")
            table.to_csv(path, index=False)
            exp.log_artifact(str(path))
print("Conclusion run closed:", conclusion_run, experimental_status)
print("Refitted experimental model:", model_name, refit_registered.version_name)

## 13. Inspect What Was Preserved

The registered refit pipeline should expose prediction and probability methods with the raw feature signature. Inspect its upstream lineage: the refit model should reference the refit Dataset, which references the six Feature Views and source data. The refit Experiment run should also be linked to this model version.

Explore the printed run URL in Snowsight and its linked model. The SQL/Python lineage view below is evidence to inspect, not a claimed result before execution. Registration may create an initial default version automatically; this notebook does not explicitly set defaults or champion aliases.

**Next phase:** extract only the modelling decisions and code that proved useful into a repeatable training workflow. Scheduling, production promotion, serving, and monitoring remain later steps.

In [ ]:
display(refit_registered.show_functions())
display(refit_registered.lineage(direction="upstream"))
display(refit_dataset.lineage(direction="upstream"))
session.lineage.trace(
    f"{database_name}.{dev_schema}.{model_name}", "MODEL", object_version=refit_registered.version_name,
    direction="upstream", distance=3,
).show()
print("Experiment:", f"{database_name}.{dev_schema}.{experiment_name}", "Runs:", final_run, conclusion_run)